### Homework #1 Alex Khvatov

In [ ]:
MODEL = "meta-llama-3.1-8b-instruct"

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

#### Q1 How many lesson pages

How many lesson pages are in the dataset?

In [ ]:
len(documents)
#72


#### Q2. Indexing and searching

In [ ]:
documents[0]

In [ ]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index = build_index(documents)

In [ ]:
from typing import List
def search_index(question: str) -> list[str]:
    '''Search the course content for the question.
    '''
    return index.search(question, 
    boost_dict={"content": 3.0, "filename": 0.5},
    num_results=5)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"
search_index(question)

#01-agentic-rag/lessons/14-agentic-loop.md

#### Q3. RAG

In [ ]:
from rag_helper_hw import RAGBaseHW1
from lmstudio import get_lmstudio_client

openai_client = get_lmstudio_client()

rag_helper = RAGBaseHW1(
    index,
    openai_client,
    model=MODEL
)

In [ ]:
response = rag_helper.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Input tokens: {response.usage.input_tokens}")
print(response.output_text)

#Input tokens 4095

#### Q4. Chunking

In [ ]:
from gitsource import chunk_documents

chunked_documents = chunk_documents(documents, size=2000, step=1000)
len(chunked_documents)
#295


#### Q5. RAG with chunking

In [ ]:
chunked_documents[:2]

In [ ]:
chunked_index =  build_index(chunked_documents)
rag_helper_chunked = RAGBaseHW1(
    chunked_index,
    openai_client,
    model=MODEL
)
response = rag_helper_chunked.rag(question)
print(response.output_text)
print(f"Input tokens: {response.usage.input_tokens}")

#Input tokens: 2271
#A 3x fewer


#### Q6. Turning it into an agent

In [ ]:
INSTRUCTIONS = """You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering."""

In [ ]:
from typing import List
def search_chunked_index(question: str) -> list[str]:
    '''Search the course content for the question.
    '''
    return chunked_index.search(question, 
    boost_dict={"content": 3.0, "filename": 0.5},
    num_results=5)

In [ ]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from toyaikit.chat import IPythonChatInterface


agent_tools = Tools()
agent_tools.add_tool(search_chunked_index)

chat_interface=IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
llm_client = OpenAIClient(model = MODEL,
        client=openai_client,)

runner = OpenAIResponsesRunner(
    llm_client=llm_client,
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
)



In [ ]:
agent_tools.get_tools()

In [ ]:
result = runner.loop(question, callback=callback)